# Lecture 2/ GV: Databases and downloads

## Tasks for today's lecture:
* Overview of Databases available for research and strategy development
* Methods to access data (pandas-datareader, wrds, nasdaq-data-link, others)
* Data organization and operations using one-, two-, and multidimensional arrays
* Storing the data: binary and text formats/ reading the data
* Download factors/ CRSP data for all SP500 components (or 500 largest stocks)/ save for further analysis
* Compute some variables to be used as features in the subsequent analysis: momentum, reversal, volatility, factor betas
* ### At the end we will pack useful code into a function so that you can run it/ save all the data


<div class="alert alert-block alert-info">
<b>Hint:</b> Note that some packages for accessing databases may be missing in the standard Anaconda distribution. 
</div>
 
You can install then using conda / pip from the Terminal/ Command window: 
* `pip install nasdaq-data-link` 
* `pip install pandas-datareader` 
* `pip install wrds`
* ...

The respective packages typically have a very extensive usage instructions on their web-sites:
* pandas-datareader:
    * https://pandas-datareader.readthedocs.io/en/latest/
* Nasdaq Data Link: 
    * https://data.nasdaq.com/tools/python 
* WRDS: 
    * https://wrds-www.wharton.upenn.edu/pages/support/programming-wrds/programming-python/python-from-your-computer/
    * https://pypi.org/project/wrds/
* Twitter (e.g., tweepy):
    * https://docs.tweepy.org/en/v4.12.1/getting_started.html
* Bloomberg 
    * https://github.com/msitt/blpapi-python 
    * https://www.bloomberg.com/professional/support/api-library/ 
    
**Some useful info:**
* Getting S&P500 components from WRDS: https://wrds-www.wharton.upenn.edu/pages/support/applications/programming-examples-and-other-topics/sp-500-datasets-and-constituents/
* The permno-cusip-secid link is available to download from https://osf.io/btvdh/

In [7]:
import platform
my_system = platform.uname()
print(f'My PC node: {my_system.node.lower()}')

My PC node: huangyikaidemacbook-pro.local


In [8]:
# after we install all the packages, import all of them for the use in today's lecture!
import platform
my_system = platform.uname()
print(f'My PC node: {my_system.node.lower()}')

# database access
import pandas_datareader as web
import nasdaqdatalink
import wrds as wrds

# storage and operations
import pandas as pd
import numpy as np
import datetime
from pathlib import Path
import joblib

from statsmodels.regression.rolling import RollingOLS

from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count
if cpu_count() > 30:
    CPUUsed = 15
else:
    CPUUsed = cpu_count() - 2

My PC node: huangyikaidemacbook-pro.local


In [9]:
DATA_FROM_LIMIT = datetime.datetime(2015,1,1)
# we can specify some options depending on your computer
class Options:
    if my_system.node.lower()=='mac': # GREGS MACHINE - do not change please
        path = Path('../DataQT2026')
        wrds_name = 'gvfs'
        nasdaqlink_key = 'XXX'
    elif (my_system.node.lower() == 'YOU CAN PUT HERE YOUR NODE'):
        path = Path(r'/.../DataQT2026/')
        wrds_name = 'XXX'
        nasdaqlink_key = 'XXX'
        pass
print('Data Path: ',Options.path)

# some file names
class FileNames:
    fn_csv_factors   = Options.path / 'ff_fact.csv'
    fn_excel_factors = Options.path / 'ff_fact.xls'

    fn_crsp     = Options.path / 'crsp.parquet'
    fn_stock_features_labels = Options.path / 'stock_features.parquet'
    fn_option_features = Options.path / 'option_features.parquet'
    fn_ff_factors    = Options.path / 'ff_factors.parquet'

    fn_sp500comp     = Options.path / 'SP500_Index_Constitutes2024.csv'
    fn_id_link = Options.path / 'crsp_secid_link.csv'
    fn_universe = Options.path / 'permno_selection.csv'
    
############################
# save the NASDAQ Link API Key:
# nasdaqdatalink.ApiConfig.api_key = Options.nasdaqlink_key
# nasdaqdatalink.save_key(nasdaqdatalink.ApiConfig.api_key)
# nasdaqdatalink.read_key()
# print(nasdaqdatalink.ApiConfig.api_key)
    
############################

AttributeError: type object 'Options' has no attribute 'path'

In [ ]:
FileNames.fn_sp500comp
sp500_comp =pd.read_csv(FileNames.fn_sp500comp)
sp500_comp

In [ ]:
dp = wrds.Connection(wrds_username=Options.wrds_name)

# Pandas datareader -> factors

Read about pandas-datareader:
https://pandas-datareader.readthedocs.io/en/latest/

A very nice tool to get factor returns and Fama-French test portfolios. Let us see how we can download the data and 
explore it a bit. We will also save factors for further use. 

In [ ]:
# let us check the available datasets
sets = web.famafrench.get_available_datasets()
sets

In [ ]:
startdt = datetime.datetime(2010,1,1)

#
d1 = web.DataReader('F-F_Research_Data_Factors_daily','famafrench',start=startdt)
d1

In [ ]:
# from the datasets let us download the FF factors and put them into one DataFrame
startdt = datetime.datetime(2010,1,1)

#
d1 = web.DataReader('F-F_Research_Data_Factors_daily','famafrench',start=startdt)
d2 = web.DataReader('F-F_Momentum_Factor_daily','famafrench',start=startdt)
d3 = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily','famafrench',start=startdt)
# #
ff = d1[0]/100
ff = ff.join(d2[0]/100, how = 'left')
ff5 = d3[0]/100
ff = ff.join(ff5.loc[:,['RMW','CMA']], how = 'left')
# #
ff.columns = [z.lower().strip() for z in ff.columns]
ff.rename(columns = {'mkt-rf':'mktrf'}, inplace = True)
# #
ff = ff.loc[:,['mktrf', 'smb', 'hml', 'mom', 'rmw', 'cma','rf']]

In [ ]:
ff.head()

In [ ]:
ff.to_parquet(FileNames.fn_ff_factors)

In [ ]:
ff = pd.read_parquet(FileNames.fn_ff_factors)
ff.head()

### Now 
* compute the average risk premiums and Sharpe for each factor (p.a.)
* compute cumulative ROI of each factor from 1996 and plot it

In [ ]:
mn = ff.mean(axis=0)*252*100
vol= ff.std()*np.sqrt(252)*100
sr = mn/vol
sr

### Compute the rolling window SR (e.g., using past 252 days), and plot it

In [ ]:
mn = ff.rolling(window = 252).mean()*252
vol = ff.rolling(window = 252).std()*np.sqrt(252)
sr = mn/vol
sr[['mktrf','smb','hml']].plot()
sr[['mktrf','smb','hml']].corr()

### Let us 
check some useful operations with files -- reading, saving, etc. 

In [ ]:
# save the factors/ play with formats
# to csv
# ff.to_csv(FileNames.fn_csv_factors)

# # from csv
# temp = pd.read_csv(FileNames.fn_csv_factors)
# temp = temp.set_index('Date')

# # # to Excel
# writer = pd.ExcelWriter(FileNames.fn_excel_factors)
# ff.to_excel(writer,'ff1')
# ff.to_excel(writer,'ff2')
# writer.save()

# ff.to_hdf(FileNames.fn_h5_factors, key = 'factd/ff', mode= 'a',data_columns = True, complib = 'zlib')

# read back from HDF
# ff1 = pd.read_hdf(FileNames.fn_h5_factors,'factd/ff')


# with pd.HDFStore(FileNames.fn_h5_factors) as store:
#     store['factd/ff']   = ff
#     print(store.keys())
#     ff2 = store['factd/ff']

In [ ]:
nasdaqdatalink.get_table('QDL/BCHAIN', date='2016-07-17', code='TVTVR')

# WRDS

WRDS is the major source of data for asset pricing and corporate finance! Let us look first at the WRDS web-site, and then we will check how to get data from it using Python API. 

There are several ways, as usual, to get the data: 
1. Directly using sql query
2. Using Pandas functionality!


Note that we will download some data on SP500 components. 
We don't have access to the dsp500list table at FS, but I have downloaded it and saved under fn_sp500comp name using: 
#data = db.raw_sql('select a.*from crsp.dsp500list as a;')
#data.to_csv(fn_sp500comp)


In [ ]:
# establish WRDS connection
# db = wrds.Connection()
# db.create_pgpass_file()

# later you can use 
# db = wrds.Connection(wrds_username=Options.wrds_name)


# WRDS API has some useful methods: 
# db.close()
# db.connection()
# db.describe_table()
# db.get_table()
# db.list_tables()
# db.raw_sql()
# db.get_row_count()
# db.list_libraries()

# For exmaple:
# db.list_libraries()
# db.describe_table(library='compm', table='idxcst_his')
# db.list_tables(library='crsp')
# db.describe_table(library='crsp', table='stocknames')

# 2. downloading whole table or number of obs 
# stocknames = db.get_table(library='crsp', table='stocknames', obs=10)
# stocknames

# 3. Using SQL
# data = db.raw_sql("select * from djones.djdaily")

# with parameters
# parm = {'tickers': ('0015B', '0030B', '0032A', '0033A', '0038A')}
# data = db.raw_sql('SELECT datadate,gvkey,cusip,tic FROM comp.funda WHERE tic in %(tickers)s', params=parm)
# data

In [ ]:
# data

## Let us learn to do the following:
* get the SP500 components
* load the returns, shares outstanding and prices for SP500 companies
* compute the value weights for the index
* save the data in a suitable format -- we discuss which one

In [ ]:
# db = wrds.Connection(wrds_username=Options.wrds_name)
# ****************************************************
# one way to get the SP500 + other index constituents  **********
# ****************************************************
# db.describe_table(library='compm', table='idxcst_his')
#      name  nullable        type
# 0   gvkey      True  VARCHAR(6)
# 1     iid      True  VARCHAR(3)
# 2  gvkeyx      True  VARCHAR(6)
# 3    from      True        DATE
# 4    thru      True        DATE
# idx_comp = db.get_table('compm', 'idxcst_his')

# db.describe_table(library='crsp', table='ccmxpf_lnkhist')
# 0      gvkey      True        VARCHAR(6)
# 1   linkprim      True        VARCHAR(1)
# 2       liid      True        VARCHAR(3)
# 3   linktype      True        VARCHAR(2)
# 4    lpermno      True  DOUBLE PRECISION
# 5    lpermco      True  DOUBLE PRECISION
# 6     linkdt      True              DATE
# 7  linkenddt      True              DATE


# ****************************************************
# another way to get the SP500 (ONLY) constituents ****
# ****************************************************
# db.describe_table(library='crsp', table='dsp500list')
# #      name  nullable              type
# # 0  permno      True  DOUBLE PRECISION
# # 1   start      True              DATE
# # 2  ending      True              DATE

# dsp500list = pd.read_csv(fn_sp500comp)
# dsp500list
# ****************************************************
# db = wrds.Connection(wrds_username=Options.wrds_name)
# db.describe_table(library='crsp', table='stocknames')
# nm = db.get_table(library='crsp', table='stocknames')
# nm.to_csv('stocknames.csv')
# nm.columns

Assume that we have the SP500 components from elsewhere, let's create some SQL queries to get the data from WRDS for these components; use components from 2015. For the class, let us download and process data for the first 20 PERMNOs only. You will run the query for the whole SP500 universe at home -- it will take some time to compute. We also can load the linking table to match CRSP and OptionMetrics (for later). 

In [ ]:
# read the sp500 composition and make the string from PERMNO for SQL
comp = pd.read_csv(FileNames.fn_sp500comp)
comp['start']  = pd.to_datetime(comp['start'])
comp['ending'] = pd.to_datetime(comp['ending'])
comp
comp = comp.loc[comp.ending>DATA_FROM_LIMIT,:] 

print(comp.head(),'\n')
np.unique(comp.permno).shape

universe = list(pd.read_csv(FileNames.fn_universe)[:10].permno)

print(universe,'\n')

crsp_secid_link = pd.read_csv(FileNames.fn_id_link)
crsp_secid_link.columns = [z.lower().strip() for z in crsp_secid_link.columns]
crsp_secid_link['sdate'] = pd.to_datetime(crsp_secid_link['sdate'])
crsp_secid_link['edate'] = pd.to_datetime(crsp_secid_link['edate'])
crsp_secid_link = crsp_secid_link[crsp_secid_link['edate']>DATA_FROM_LIMIT]
print(crsp_secid_link)

In [ ]:
len(tuple(universe))

In [ ]:
db = wrds.Connection(wrds_username=Options.wrds_name)

In [ ]:
# load the returns, shares outstanding and prices for SP500 companies (from 2015)
# create a suitable dataframes/ compute market cap/ save
sql_wrds = """
select distinct date, permno, cusip, ret, abs(prc) as prc, 
    shrout/1000 as shrout, abs(prc)*shrout/1000 as mktcap
from crsp.dsf
where permno in %(PERMNO)s and EXTRACT(YEAR FROM date) >= %(YR)s
"""

# # specify the prm dictionary with permno
prm = {'PERMNO':tuple(universe), 'YR':2014}
#
# # get the data!
data = db.raw_sql(sql_wrds, params = prm)

data['date'] = pd.to_datetime(data['date'])

In [ ]:
data.head()

In [ ]:
cols2use = ['permno','date','prc','ret','mktcap']
data0 = data[cols2use].copy()

### Check! 
Now chech the data using a simple `describe()` method... anything strange? What about negative prices? How come? Please check how we can get negative prices and verify that we did everything correct. 

In [ ]:
data[['ret', 'prc', 'shrout', 'mktcap']].describe()

Now check for each date and each company whether it is in the index! Using this information, compute daily value weights (e.g., to be used to compute performance of a value-weighted portfolio).

At the end compute the value-weighted growth process and plot it.

## Compute some important features/ labels for our analysis

* Forward returns (1- and 5-day): it would be better to compute them on-the-fly, so we will not save them! Let us just see how we can use Pandas functionality to prepare them. 
* Momentum and reversal characteristics
* Factor betas

In [ ]:
cols2use = ['permno','date','ret','mktcap']
data0 = data[cols2use].copy()
data0.sort_values(by = ['permno','date'], inplace = True)

# let us compute 1-day return by groups
data0['fret1d'] = data0.groupby(by=['permno'])['ret'].shift(-1)

# let us compute 5-day forward return
fn_5dret = lambda x: np.exp(x.rolling(window=5, min_periods=3).sum())-1

data0['lret'] = np.log(1+data0['ret'])
data0['fret5d'] = data0.\
                    groupby(by=['permno'], group_keys=False)['lret'].\
                    apply(fn_5dret).shift(-5)

data0

In [ ]:
# let's continue with the same data0 frame and construct momentum/ reversal 
# momentum/ reversal can be defined in a number of ways, 
#e.g. momentum = total return from months - 13 to -1:
fn_mom12m = lambda x: np.exp(x.rolling(window=252, min_periods=63).sum())-1
#e.g. momentum = total return from months - 7 to -1:
fn_mom6m = lambda x: np.exp(x.rolling(window=126, min_periods=31).sum())-1
#e.g. reversal = total return for past month:
fn_rev = lambda x: np.exp(x.rolling(window=21, min_periods=15).sum())-1


data0['mom12m'] = data0.\
                    groupby(by=['permno'], group_keys=False)['lret'].\
                    apply(fn_mom12m).shift(22) 

data0['mom6m'] = data0.\
                    groupby(by=['permno'], group_keys=False)['lret'].\
                    apply(fn_mom6m).shift(22) 

data0['rev'] = data0.\
                    groupby(by=['permno'], group_keys=False)['lret'].\
                    apply(fn_rev) 
data0.tail(10)

# we might want to drop log returns if we do not need them anymore 
# data0.drop(columns = 'lret', inplace = True)



In [ ]:
# let us create factor betas using 252 daily excess returns/ factor realizations
# because such an operation is time-consuming, we will also parallelize it later on 
# create a slimmer copy for work 
cols2use = ['permno','date','ret']
data1 = data0[cols2use].copy().set_index(['date','permno']).sort_index()

df_ret = data1['ret'].unstack()

df_ret
# load factors and merge them with the data file: 
ff = pd.read_parquet(FileNames.fn_ff_factors)

# subtract risk-free rate from returns to get excess returns 
idx = df_ret.index
df_ret = df_ret.subtract(ff.loc[idx,'rf'], axis=0)

# merge to include factors 
df_ret = df_ret.merge(ff.loc[idx,['mktrf','smb','hml','mom','rmw','cma']], on = 'date')


print(df_ret.tail())

In [ ]:
# there are many ways to run a regression in Python... 
# we want to estimate a regression evey day, so we would use RollingOLS.from_formula
# e.g., get data for one permno

factors = ['mktrf','smb','hml','mom']

permno = universe[0]
temp = df_ret[[permno,*factors]]
temp = temp.rename(columns = {permno:'ret'})

formula = 'ret~1+mktrf+smb+hml+mom'
window = 252
min_n = 126
rres = RollingOLS.from_formula(formula, 
                               data=temp, 
                               window=window, 
                               min_nobs=min_n,
                               missing='drop').fit(params_only=False)

print(rres.params.iloc[:,1:])
print((rres.mse_resid*252)**0.5)

In [ ]:
# let us write the function that accepts some parameters for flexible factor models 
def get_beta_bypermno(df, permno=None, window=252, min_n=int(252/2), factor_model='ff4'):

    if (permno is not None) and (permno in df.columns): 
        df = df.rename(columns={permno: 'ret'})
    else:
        print(f'permno {permno} not in the data')
             

    if factor_model == 'capm':
        formula = f'ret ~ 1 +  mktrf'
    elif factor_model == 'ff3':
        formula = f'ret ~ 1 +  mktrf + smb + hml'
    elif factor_model == 'ff4':
        formula = f'ret ~ 1 +  mktrf + smb + hml + mom'
    elif factor_model == 'ff5':
        formula = f'ret ~ 1 +  mktrf + smb + hml + rmw + cma'
    else:
        formula = f'ret ~ 1'
        pass

    try:
        rres = RollingOLS.from_formula(formula, 
                                       data=df, 
                                       window=window, 
                                       min_nobs=min_n,
                                       missing='drop').fit(params_only=False)
        temp_beta = rres.params.iloc[:,1:]
        temp_beta.columns = [f'beta_{z}' for z in temp_beta.columns]
        temp_beta = temp_beta.join((rres.mse_resid.to_frame(f'idvar_{factor_model}')*252))
        temp_beta['permno'] = permno
        temp_beta = temp_beta.dropna(axis=1, how ='all')
        return temp_beta.reset_index()
    except:
        pass

In [ ]:
factors = ['mktrf','smb','hml','mom']
permno = universe[0]
# run for one permno 
res0 = get_beta_bypermno(df_ret[[permno,*factors]], permno = permno)
print(res0)

# run for a number of stocks 
res = [get_beta_bypermno(df_ret[[permno,*factors]], permno = permno) for permno in universe]
temp = pd.concat(res)
print(temp)

In [ ]:
# because it is still slow for a large number of stocks, we can parallelize it 
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count
if cpu_count() > 30:
    CPUUsed = 15
else:
    CPUUsed = cpu_count() - 2

with Parallel(n_jobs=CPUUsed) as parallel:
        res = parallel(delayed(get_beta_bypermno)(df_ret[[permno, *factors]],
                                          permno,
                                          window = 252,
                                          min_n = 126,
                                          factor_model = 'ff4') for permno in universe)


In [ ]:
len(res)

In [ ]:
temp = pd.concat(res)
temp = temp.set_index(['permno','date']).sort_index().dropna()
print(temp)

# Function definition to download all the data -> save

Let us pack everything we programmed in a couple of functions, so that we can run them anytime. Before class, I have written a short version of such functions, and will extend/ enhance them during and after the class. You should do the same independently!

In [5]:
spx_composition_from = 2014
comp = pd.read_csv(FileNames.fn_sp500comp)
comp['start']  = pd.to_datetime(comp['start'])
comp['ending'] = pd.to_datetime(comp['ending'])
comp = comp.loc[comp['ending'].dt.year>=spx_composition_from,:]
universe_permno = tuple(set(comp['permno']))
UNIVERSE_PERMNO = f'AND a.permno in {universe_permno}'
print(f'Restricting to SP500 universe: {len(universe_permno)} stocks')
UNIVERSE_PERMNO

NameError: name 'FileNames' is not defined

In [ ]:
def prepare_save_rets(load_factors = True, factors_from = 2014,
                     load_crsp_data = True, crsp_from = 2014,
                     restrict_universe = True, 
                     spx_composition_from = 2014,
                     year_to = 2025):
    import time

    UNIVERSE_PERMNO = ''
    if restrict_universe:
        # read the sp500 composition and make the string from PERMNO for SQL
        comp = pd.read_csv(FileNames.fn_sp500comp)
        comp['start']  = pd.to_datetime(comp['start'])
        comp['ending'] = pd.to_datetime(comp['ending'])
        comp = comp.loc[comp['ending'].dt.year>=spx_composition_from,:]
        universe_permno = tuple(set(comp['permno']))
        UNIVERSE_PERMNO = f'AND a.permno in {universe_permno}'
        print(f'Restricting to SP500 universe: {len(universe_permno)} stocks')
        pass

    if load_factors:
        startdt = datetime.datetime(factors_from,1,1)

        d1 = web.DataReader('F-F_Research_Data_Factors_daily','famafrench',start=startdt)
        d2 = web.DataReader('F-F_Momentum_Factor_daily','famafrench',start=startdt)
        d3 = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily','famafrench',start=startdt)

        ff = d1[0]/100
        ff = ff.join(d2[0]/100, how = 'left')
        ff5 = d3[0]/100
        ff = ff.join(ff5.loc[:,['RMW','CMA']], how = 'left')

        ff.columns = [z.lower().strip() for z in ff.columns]
        ff.rename(columns = {'mkt-rf':'mktrf'}, inplace = True)

        ff = ff.loc[:,['mktrf', 'smb', 'hml', 'mom', 'rmw', 'cma','rf']]
        ff.to_parquet(FileNames.fn_ff_factors)

        print(f'Saved FF factors')
        pass

    if load_crsp_data:
        # get the data!
        db = wrds.Connection(wrds_username=Options.wrds_name)
        
        start = time.time()
        
        data_crsp = pd.DataFrame()
        for YR in range(crsp_from,year_to+1):
            try:
                sql_wrds = f"""
                select distinct a.date, a.permno, a.cusip, 
                a.ret, abs(a.prc) as close, a.openprc as open, a.askhi as high, a.bidlo as low, 
                a.shrout, abs(a.prc)*a.shrout/1000 as mktcap, a.vol
                from crsp.dsf as a
                where
                EXTRACT(YEAR FROM a.date) = {YR}
                {UNIVERSE_PERMNO}
                ORDER BY a.permno, a.date
                """
                data_crsp = pd.concat([data_crsp,db.raw_sql(sql_wrds)], axis = 0)
                print(f'finished downloading CRSP {YR} in {(time.time()-start)/60:.2f} minutes')
                pass
            except:
                print(f'No data for year {YR}')
                pass
        
        data_crsp['date'] = pd.to_datetime(data_crsp['date'])
        db.close()

        data_crsp = data_crsp.sort_values(by = ['permno','date'])
        data_crsp['mktcap'] = data_crsp.groupby(['permno'])['mktcap'].shift(1)
        data_crsp['vw'] = data_crsp['mktcap']/data_crsp.groupby(['date'])['mktcap'].transform('sum')

        data_crsp.to_parquet(FileNames.fn_crsp)
        print(f'Saved CRSP data')
        pass
    pass


def get_beta_bypermno(df, permno=None, window=252, min_n=int(252/2), factor_model='ff4'):

    if (permno is not None) and (permno in df.columns): 
        df = df.rename(columns={permno: 'ret'})
    else:
        print(f'permno {permno} not in the data')
             

    if factor_model == 'capm':
        formula = f'ret ~ 1 +  mktrf'
    elif factor_model == 'ff3':
        formula = f'ret ~ 1 +  mktrf + smb + hml'
    elif factor_model == 'ff4':
        formula = f'ret ~ 1 +  mktrf + smb + hml + mom'
    elif factor_model == 'ff5':
        formula = f'ret ~ 1 +  mktrf + smb + hml + rmw + cma'
    else:
        formula = f'ret ~ 1'
        pass

    try:
        rres = RollingOLS.from_formula(formula, 
                                       data=df, 
                                       window=window, 
                                       min_nobs=min_n,
                                       missing='drop').fit(params_only=False)
        temp_beta = rres.params.iloc[:,1:]
        temp_beta.columns = [f'beta_{z}' for z in temp_beta.columns]
        temp_beta = temp_beta.join((rres.mse_resid.to_frame(f'idvar_{factor_model}')*252))
        temp_beta['permno'] = permno
        temp_beta = temp_beta.dropna(axis=1, how ='all')
        return temp_beta.reset_index()
    except:
        pass
    pass


def compute_features_labels(data, 
                            ff,
                            factor_model = 'ff4',
                            SaveResults = True):
    
    res_dict = {}
    
    cols2use = ['permno','date','ret','mktcap']
    data0 = data.reset_index()[cols2use].copy()
    data0.sort_values(by = ['permno','date'], inplace = True)
    data0['lret'] = np.log(1+data0['ret'])
    
    # momentum/ reversal can be defined in a number of ways, 
    #e.g. momentum = total return from months - 13 to -1:
    fn_mom12m = lambda x: np.exp(x.rolling(window=252, min_periods=63).sum())-1
    #e.g. momentum = total return from months - 7 to -1:
    fn_mom6m = lambda x: np.exp(x.rolling(window=126, min_periods=31).sum())-1
    #e.g. reversal = total return for past month:
    fn_rev = lambda x: np.exp(x.rolling(window=21, min_periods=15).sum())-1
    # let us compute 5-day forward return
    fn_5dret = lambda x: np.exp(x.rolling(window=5, min_periods=3).sum())-1

    # let us compute 1-day return by groups
    data0['fret1d'] = data0.groupby(by=['permno'])['ret'].shift(-1)

    data0['fret5d'] = data0.\
                        groupby(by=['permno'], group_keys=False)['lret'].\
                        apply(fn_5dret).shift(-5)

    data0['mom12m'] = data0.\
                        groupby(by=['permno'], group_keys=True)['lret'].\
                        apply(fn_mom12m).shift(22).values

    data0['mom6m'] = data0.\
                        groupby(by=['permno'], group_keys=True)['lret'].\
                        apply(fn_mom6m).shift(22).values

    data0['rev1m'] = data0.\
                        groupby(by=['permno'], group_keys=True)['lret'].\
                        apply(fn_rev).values
    
    data0.drop(columns = ['lret'], inplace = True)
    # mom_rev 
    print('Computed momentum/ reversal/ forward returns')
    momrev = {'momrev':['mom12m','mom6m','rev1m']}
    res_dict.update(momrev)
    fwdret = {'fwdret':['fret1d','fret5d']}
    res_dict.update(fwdret)
    
    # factor betas 
    # create a slimmer copy for work 
    cols2use = ['permno','date','ret']
    data1 = data0[cols2use].copy().set_index(['date','permno']).sort_index()

    df_ret = data1['ret'].unstack()
    universe = list(df_ret.columns)

    # subtract risk-free rate from returns to get excess returns 
    idx = df_ret.index
    df_ret = df_ret.subtract(ff.loc[idx,'rf'], axis=0)

    # merge to include factors 
    factors = ['mktrf','smb','hml','mom','rmw','cma']
    df_ret = df_ret.merge(ff.loc[idx,factors], on = 'date')
    
    with Parallel(n_jobs=CPUUsed) as parallel:
        res = parallel(delayed(get_beta_bypermno)(df_ret[[permno, *factors]],
                                          permno,
                                          window = 252,
                                          min_n = 126,
                                          factor_model = factor_model) for permno in universe)
        pass
    
    temp = pd.concat(res)
    temp = temp.set_index(['permno','date']).sort_index().dropna()
    
    ##
    print('Computed factor betas')
    facts = {'fmodels':list(temp.columns)}
    res_dict.update(facts)
    
    data0 = data0.set_index(['permno','date'])
    data0 = data0.merge(temp, on = ['permno','date'])
    
    if SaveResults:
        data0.to_parquet(FileNames.fn_stock_features_labels)
        print('*'*5, 'Saved the results')
        pass

    return data0, res_dict

'''DEFINE FUNCTIONS TO LOAD THE DATA'''   
def load_ff_crsp():
    crsp = pd.read_parquet(FileNames.fn_crsp)
    ff = pd.read_parquet(FileNames.fn_ff_factors)
    return crsp, ff   


def load_features():
    features_crsp = pd.read_parquet(FileNames.fn_stock_features_labels)
    return features_crsp  

In [ ]:
import wrds as wrds

import platform
my_system = platform.uname()
print(f'My PC node: {my_system.node.lower()}')

class Options:
    if (my_system.node.lower().startswith('mac')): # GREG'S MACHINE - do not change please
        path = Path('../DataQT2026')
        wrds_name = 'gvfs'
        nasdaqlink_key = 'XXX'
    elif (my_system.node.lower() == 'huangyikaidemacbook-pro.local'):
        path = Path(r'/Users/huangyikai/Documents/FS SEM4/QT')
        wrds_name = 'yi_kai'
        nasdaqlink_key = 'XXX'
        pass
print('Data Path: ',Options.path)

# Make sue the Universe_SECID can be directly used in qeury
universe_secid = tuple(
    int(x) for x in comp_with_secid['secid'].dropna().unique()
)
SECID_LIST = universe_secid
print(type(SECID_LIST))
print(SECID_LIST[:5])


db = wrds.Connection(wrds_username=Options.wrds_name)
# a more complex example that we use to compute ATM IV and some skewness proxies

sql_wrds_1 = f"""
select date,
       secid,
       avg(impl_volatility) as aiv_otm
from (

      select date, secid, impl_volatility
      from optionm.vsurfd2015
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2016
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2017
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2018
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2019
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2020
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2021
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2022
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2023
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2024
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

      union all

      select date, secid, impl_volatility
      from optionm.vsurfd2025
      where days = 30
        and abs(delta) < 50
        and secid in {SECID_LIST}

     ) as all_years

group by date, secid
order by date, secid
"""

data_om = db.raw_sql(sql_wrds_2)


My PC node: huangyikaidemacbook-pro.local
Data Path:  /Users/huangyikai/Documents/FS SEM4/QT
<class 'tuple'>
(108505, 107525, 109181, 105785, 104049)
Loading library list...
Done


# run data preparation 

In [ ]:
prepare_save_rets(load_factors = True, factors_from = 2014,
                     load_crsp_data = True, crsp_from = 2014,
                     spx_composition_from = 2014,
                     restrict_universe = True)

In [ ]:
crsp, ff = load_ff_crsp()
crsp.head()

In [ ]:
res, res_dict = compute_features_labels(data = crsp, 
                            ff = ff,
                            factor_model = 'ff4',
                            SaveResults = True)
print(res_dict)

In [ ]:
crsp, ff = load_ff_crsp()
features_crsp = load_features()
print(features_crsp.columns)

In [ ]:
features_crsp